## Google Colab Classification Sweep Notebook

This notebook runs one model across every combination of named data and training configuration variants from within Google Colab.

In [ ]:
# Initial imports
import sys
import os
from google.colab import drive

In [ ]:
# Clone the repository to the Colab environment
!git clone https://github.com/gemixin/touch-ex

In [ ]:
# Set up the repository path and add it to the Python path
repo_path = "/content/touch-ex"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
os.chdir(repo_path)

In [ ]:
# Mount Google Drive so experiment outputs persist after the Colab session ends
drive.mount("/content/drive")

# Set up directories for experiment outputs
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/touch-ex"
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# Install the project dependencies. Colab's existing CUDA-enabled PyTorch is used
# when it already satisfies the torch and torchvision requirements
%pip install -q -r requirements.txt

In [ ]:
import json

from models.experiments import classify_sweep


# --- Configurable parameters --- #

# Chosen model type for sweep experiments
# Choose from 'baseline', 'resnet18', 'efficientnet_b0', 'vit_b_16', 'deit_tiny', or
# 't3_tiny'
MODEL_TYPE = "resnet18"

# Target label for classification
# Choose from 'object', 'object_region', 'force_level', or 'motion'
TARGET_LABEL = "object"

# Experiment name for tracking results
EXPERIMENT_NAME = "resnet18_aug_sweep"

# Randomisation settings
SEED = 129
DETERMINISTIC = True

# Set to True to train only the classifier of pretrained models
# Baseline models are always trained end-to-end
FREEZE_BACKBONE = False

# Load the SSVTP color jitter settings from the JSON file
with open(
    os.path.join(repo_path, "configs/ssvtp_color_jitter_settings.json"),
    "r",
    encoding="utf-8",
) as file:
    ssvtp_color_jitter = json.load(file)["color_jitter"]

# Each data variant is combined with every training variant below
# Values override keys in the chosen data config file
DATA_CONFIG_VARIANTS = {
    "none": {
        "transform_name": "center_crop_224",
        "train_augmentations": {
            "color_jitter": None,
            "horizontal_flip": None,
            "random_resized_crop": False,
        },
    },
    "color_jitter": {
        "transform_name": "center_crop_224",
        "train_augmentations": {
            "color_jitter": ssvtp_color_jitter,
            "horizontal_flip": None,
            "random_resized_crop": False,
        },
    },
    "random_resized_crop": {
        "transform_name": "center_crop_224",
        "train_augmentations": {
            "color_jitter": None,
            "horizontal_flip": None,
            "random_resized_crop": True,
        },
    },
    "both": {
        "transform_name": "center_crop_224",
        "train_augmentations": {
            "color_jitter": ssvtp_color_jitter,
            "horizontal_flip": None,
            "random_resized_crop": True,
        },
    },
}

# Each training variant is combined with every data variant above
# Values override keys in the chosen training config file
TRAIN_CONFIG_VARIANTS = {
    "default": {},
}

# t-SNE feature plot settings
PLOT_TSNE = False
TSNE_MAX_SAMPLES = -1

# Paths for configuration files
DATA_CONFIG_PATH = os.path.join(repo_path, "configs/default_data_config.json")
TRAIN_CONFIG_PATH = os.path.join(
    repo_path,
    "configs/frozen_train_config.json"
    if FREEZE_BACKBONE
    else "configs/finetuned_train_config.json",
)
BASELINE_TRAIN_CONFIG_PATH = os.path.join(repo_path, "configs/baseline_train_config.json")

# --- Train and evaluate the configuration sweep --- #

models, histories, results = classify_sweep(
    model_type=MODEL_TYPE,
    data_config_variants=DATA_CONFIG_VARIANTS,
    train_config_variants=TRAIN_CONFIG_VARIANTS,
    target_label=TARGET_LABEL,
    experiment_name=EXPERIMENT_NAME,
    seed=SEED,
    deterministic=DETERMINISTIC,
    freeze_backbone=FREEZE_BACKBONE,
    plot_tsne=PLOT_TSNE,
    tsne_max_samples=TSNE_MAX_SAMPLES,
    data_config_path=DATA_CONFIG_PATH,
    train_config_path=TRAIN_CONFIG_PATH,
    baseline_train_config_path=BASELINE_TRAIN_CONFIG_PATH,
    results_dir=RESULTS_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
)